# Machine Learning Pipeline

This notebook builds a machine learning workflow for the Adult Income dataset, from data preparation to model evaluation.

In [14]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline , Pipeline
from sklearn.preprocessing import LabelEncoder 

## 1. Load the Dataset

Load the dataset into a Pandas DataFrame so it can be inspected and prepared for training.

In [2]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\archive (14).zip")

## 2. Explore the Dataset

Display the first rows to understand the columns and the structure of the data.

In [3]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


### View the Last Rows

Display the last rows as another quick check of the dataset.

In [4]:
df.tail()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32560,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


## 3. Check for Duplicate Rows

Count duplicated rows before removing them.

In [5]:
df.duplicated().sum()

np.int64(24)

### Remove Duplicates

Remove duplicate records so the same row is not repeated in the dataset.

In [6]:
df = df.drop_duplicates()

In [7]:
df.duplicated().sum()

np.int64(0)

## 4. Inspect Question-Mark Values

Check object columns for values represented by `?`, which may indicate missing data.

In [8]:
for column in df.select_dtypes(include="object").columns:
    count = (df[column].astype(str).str.strip() == "?" ).sum()
    if count > 0:
        print(column , count)


workclass 1836
occupation 1843
native.country 582


### Convert Missing-Value Markers

Replace `?` values with `NaN` so missing values can be handled consistently.

In [9]:
df = df.replace(r'^\s*\?\s*$' , np.nan , regex=True)
print(df.isnull().sum()[df.isnull().sum() > 0]) 

workclass         1836
occupation        1843
native.country     582
dtype: int64


### Inspect the Cleaned Data

Display a sample of the data after converting the missing-value markers.

In [10]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,NaN,77053,HS-grad,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,NaN,186061,Some-college,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


### Count Missing Values

Check which columns contain missing values.

In [11]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     582
income               0
dtype: int64

## 5. Feature Engineering

Create an age-group feature from the existing age column to provide an additional input feature.

In [12]:
# 1. Create a copy
df = df.copy()

# 2. Create age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 50, 100],
    labels=['young', 'adult', 'middle_age', 'older']
)

# 3. Net capital
df['capital_net'] = (
    df['capital.gain'] - df['capital.loss']
)

# 4. Has capital gain or not
df['has_capital_gain'] = (
    df['capital.gain'] > 0
).astype(int)

# 5. Categorize working hours
df['hours_category'] = pd.cut(
    df['hours.per.week'],
    bins=[0, 20, 40, 60, float('inf')],
    labels=['low', 'normal', 'high', 'very_high']
)

# Check the new features
print(df[['age_group', 'capital_net',
          'has_capital_gain', 'hours_category']].head())

print(df.shape)

    age_group  capital_net  has_capital_gain hours_category
0       older        -4356                 0         normal
1       older        -4356                 0            low
2       older        -4356                 0         normal
3       older        -3900                 0         normal
4  middle_age        -3900                 0         normal
(32537, 19)


## 6. Separate Features and Target

Use `income` as the target column (`y`) and the remaining columns as input features (`X`).

In [15]:
# Separate features and target
X = df.drop("income", axis=1)
y = df["income"]

# Encode the target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print(X.shape)
print(pd.Series(y).value_counts())


(32537, 18)
0    24698
1     7839
Name: count, dtype: int64


## 7. Import Pipeline Tools

Import the tools needed for preprocessing, feature selection, model training, tuning, and evaluation.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 8. Split Training and Test Data

Split the dataset into training and test sets. Stratification helps preserve the target-class proportions in both sets.

In [17]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


(22775, 18)
(9762, 18)


## 9. Build the Preprocessing Steps

Identify numeric and categorical columns. Fill missing numeric values with the median and scale them; fill missing categorical values with the most frequent value and encode them.

In [18]:
# Select numeric and categorical columns
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include="object").columns

# Clean and scale numeric columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Clean and encode categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply preprocessing to the right columns
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


## 10. Train and Tune the Model

Combine preprocessing, feature selection, and a Random Forest classifier into one pipeline. `GridSearchCV` tests the selected parameter combinations using cross-validation.

In [19]:
# Build the machine learning pipeline
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(score_func=mutual_info_classif)),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(random_state=42))
])

# Try different settings and choose the best one
param_grid = {
    "feature_selection__k": [10, "all"],
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)


c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:68: UserWarning: Clustering metrics expects discrete values b

Best parameters: {'feature_selection__k': 'all', 'model__max_depth': None, 'model__n_estimators': 200}
Best cross-validation score: 0.8502302955494642


## 11. Evaluate the Best Model

Use the best model found by cross-validation to predict the held-out test set, then report accuracy, classification metrics, and the confusion matrix.

In [20]:
# Evaluate the best model on the test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", accuracy)
print("\\nClassification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 0.8462405244826879
\nClassification report:
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      7410
           1       0.68      0.69      0.68      2352

    accuracy                           0.85      9762
   macro avg       0.79      0.79      0.79      9762
weighted avg       0.85      0.85      0.85      9762

Confusion matrix:
[[6633  777]
 [ 724 1628]]


## 12. Track Performance

Create a small table comparing the best cross-validation accuracy with the final test accuracy.

In [21]:
# Track model performance
results = pd.DataFrame({
    "Metric": ["Best CV Accuracy", "Test Accuracy"],
    "Score": [grid_search.best_score_, accuracy]
})

results


,Metric,Score
0,Best CV Accuracy,0.850230
1,Test Accuracy,0.846241
